# Stride vs. Pooling: three ways to compress a 28x28 digit

Side experiment on top of HW2 B.4 (`STAT453_hw02.ipynb` is untouched). Same data subset, same optimizer (Adam, lr=1e-3), same 30 epochs, same seeds -- **only the downsampling changes**.

| model | conv1 | pool | into fc | fc input |
|---|---|---|---|---|
| **Baseline** (HW2 `SmallCNN`) | stride 1 -> 26x26 | MaxPool(2) -> 13x13 | 8x13x13 | 1352 |
| **A** (pool removed) | stride 2 -> 13x13 | -- | 8x13x13 | 1352 |
| **B** (control: stride added, pool kept) | stride 2 -> 13x13 | MaxPool(2) -> 6x6 | 8x6x6 | 288 |

- **Baseline vs A**: same output size, same parameters, same initial weights. The only difference is how a 2x2 neighborhood becomes one number.
- **A vs B**: what happens when you compress *again* (13x13 -> 6x6)?

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
%matplotlib inline

torch.manual_seed(453)

## Data (same as HW2 B.4)

In [ ]:
transform = transforms.ToTensor()
train_full = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_full = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Subset for a fast homework training loop
train_idx = torch.randperm(len(train_full), generator=torch.Generator().manual_seed(453))[:4000]
test_idx = torch.randperm(len(test_full), generator=torch.Generator().manual_seed(453))[:1000]
train_subset = torch.utils.data.Subset(train_full, train_idx)
test_subset = torch.utils.data.Subset(test_full, test_idx)

train_loader = torch.utils.data.DataLoader(train_subset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_subset, batch_size=256, shuffle=False)

## The three networks

A and B subclass the baseline and only override the lines that differ. Changing `conv1.stride` in place (instead of building a new `nn.Conv2d`) keeps the initial weights identical to the baseline under the same seed.

In [ ]:
# Baseline: HW2 B.4 SmallCNN, unchanged
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,8,3,1,0)
        self.pool = nn.MaxPool2d(2,2)
        self.fc = nn.Linear(1352,10,bias = False, device= None, dtype = None)

    def forward(self, x):
        x = torch.relu(self.conv1(x))     # conv, then ReLU
        x = self.pool(x)       # pool
        x = x.view(x.size(0), -1)
        return self.fc(x)


# A: remove pooling, compress with stride instead
class StrideCNN(SmallCNN):
    def __init__(self):
        super().__init__()
        self.conv1.stride = (2, 2)   # 28x28 -> 13x13 straight out of the conv
        self.pool = nn.Identity()    # no pooling; forward() is unchanged
        # fc stays Linear(1352, 10): 8*13*13, same as baseline


# B (control): add stride but keep pooling -> compresses twice
class StridePoolCNN(SmallCNN):
    def __init__(self):
        super().__init__()
        self.conv1.stride = (2, 2)                       # 28x28 -> 13x13
        self.fc = nn.Linear(8 * 6 * 6, 10, bias=False)  # pool: 13x13 -> 6x6


MODELS = {'Baseline: stride 1 + pool': SmallCNN,
          'A: stride 2, no pool': StrideCNN,
          'B: stride 2 + pool': StridePoolCNN}

### How much does each one compress?

`conv MACs` = multiply-adds in conv1 (one 3x3 dot product per output position per channel).

In [ ]:
x, _ = next(iter(test_loader))
x = x[:1]                                    # one 1x28x28 test digit

print(f'{"":28s}{"after conv":>14s}{"after pool":>14s}{"fc input":>10s}{"params":>9s}{"conv MACs":>11s}')
for name, cls in MODELS.items():
    m = cls()
    h = torch.relu(m.conv1(x))
    p = m.pool(h)
    macs = h[0].numel() * 9
    params = sum(q.numel() for q in m.parameters())
    print(f'{name:28s}{str(tuple(h.shape[1:])):>14s}{str(tuple(p.shape[1:])):>14s}'
          f'{p[0].numel():>10,}{params:>9,}{macs:>11,}')

### What a stride-2 conv actually is

A stride-2 conv is exactly a stride-1 conv with every other row and column thrown away -- i.e. it keeps the **top-left** value of each 2x2 block and never computes the other three. Max-pooling computes all four and keeps the **largest**.

In [ ]:
w = torch.randn(8, 1, 3, 3)
full = F.conv2d(x, w, stride=1)          # (1, 8, 26, 26)
strided = F.conv2d(x, w, stride=2)       # (1, 8, 13, 13)
print(tuple(full.shape), '->', tuple(strided.shape))
print('stride-2 conv == stride-1 conv[::2, ::2]:', torch.allclose(strided, full[:, :, ::2, ::2], atol=1e-6))

## Train all three

Same training loop as HW2 B.4, wrapped in a function. Re-seeding before building the model and again before training gives every model the same initialization (where shapes match) and the same batch order. Accuracy is averaged over the last 5 epochs to smooth epoch-to-epoch jitter, then mean +/- std over seeds.

In [ ]:
def run(model_cls, seed, epochs=30):
    torch.manual_seed(seed)
    cnn = model_cls()
    torch.manual_seed(seed)        # same batch order for every model
    optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)
    test_accs, train_losses = [], []
    for epoch in range(epochs):
        cnn.train()
        running = 0.0
        for images, labels in train_loader:
            optimizer.zero_grad()
            logits = cnn(images)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()
            running += loss.item() * labels.size(0)
        train_losses.append(running / len(train_subset))

        cnn.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                preds = cnn(images).argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        test_accs.append(100 * correct / total)
    return cnn, test_accs, train_losses

In [ ]:
SEEDS = [453, 454, 455]
acc, loss, trained = {}, {}, {}
for name, cls in MODELS.items():
    runs = [run(cls, s) for s in SEEDS]
    trained[name] = runs[0][0]                       # keep one trained model per variant
    acc[name] = torch.tensor([r[1] for r in runs])   # (seeds, epochs)
    loss[name] = torch.tensor([r[2] for r in runs])
    last5 = acc[name][:, -5:].mean(dim=1)
    print(f'{name:28s} test acc {last5.mean():.1f} +/- {last5.std():.1f}%   '
          f'final train loss {loss[name][:, -1].mean():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for name in MODELS:
    epochs = range(1, acc[name].shape[1] + 1)
    axes[0].plot(epochs, acc[name].mean(0), label=name)
    axes[1].plot(epochs, loss[name].mean(0), label=name)
axes[0].set(xlabel='epoch', ylabel='test accuracy (%)', title=f'test accuracy (mean of {len(SEEDS)} seeds)')
axes[1].set(xlabel='epoch', ylabel='train loss', title='training loss', yscale='log')
axes[0].legend()
plt.tight_layout()
plt.show()

## What each compression keeps

Same trained baseline filters for every column, so the only thing that differs is the downsampling. Rows are the 4 most active channels for this digit.

In [ ]:
base = trained['Baseline: stride 1 + pool']
with torch.no_grad():
    full = torch.relu(base.conv1(x))[0]          # (8, 26, 26): conv at every position
stages = {
    'conv, stride 1\n26x26 (uncompressed)': full,
    'Baseline: max of each 2x2\n13x13': F.max_pool2d(full, 2),
    'A: stride 2 = top-left of each 2x2\n13x13': full[:, ::2, ::2],
    'B: stride 2, then max-pool\n6x6': F.max_pool2d(full[:, ::2, ::2], 2),
}
channels = full.flatten(1).amax(1).topk(4).indices.tolist()

fig, axes = plt.subplots(len(channels), len(stages) + 1, figsize=(13, 2.6 * len(channels)))
for r, c in enumerate(channels):
    axes[r, 0].imshow(x[0, 0], cmap='gray')
    axes[r, 0].set_ylabel(f'channel {c}')
    for j, (title, fmap) in enumerate(stages.items(), start=1):
        axes[r, j].imshow(fmap[c], cmap='viridis')
        if r == 0:
            axes[r, j].set_title(title, fontsize=9)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
axes[0, 0].set_title('input 28x28', fontsize=9)
plt.tight_layout()
plt.show()

## Things to look at

1. **Baseline vs A** have identical shapes and parameter counts. Whatever accuracy gap exists comes purely from *max of 2x2* vs *top-left of 2x2*. In the picture, which one keeps thin strokes that happen to land on the "wrong" pixel of a block?
2. A does **4x fewer conv multiply-adds** than the baseline for the same output size -- that is the practical reason modern nets often downsample with stride instead of pooling.
3. **A vs B**: B keeps 36 positions instead of 169 and its fc has ~4.7x fewer weights. Does its *training* loss go up too (information/capacity lost), or only its test accuracy?